In [ ]:
# ============================================================
# CELL 1 — Install requirements
# ============================================================
!pip install transformers torch scikit-learn pandas numpy

In [ ]:
# ============================================================
# CELL 2 — Load Sentiment Dataset (53K records)
# ============================================================
import pandas as pd
import numpy as np

# Load Kaggle sentiment dataset
df_sentiment = pd.read_csv("sentiment_mental_health.csv",encoding='latin-1')    # fixes UnicodeDecodeError)
print("Shape:", df_sentiment.shape)
print(df_sentiment.head())
print("\nColumns:", df_sentiment.columns.tolist())
print("\nLabel distribution:")
print(df_sentiment.iloc[:, -1].value_counts())

Shape: (53044, 3)
  Unnamed: 0                                          statement   status
0          0                                         oh my gosh  Anxiety
1          1  trouble sleeping, confused mind, restless hear...  Anxiety
2          2  All wrong, back off dear, forward doubt. Stay ...  Anxiety
3          3  I've shifted my focus to something else but I'...  Anxiety
4          4  I'm restless and restless, it's been a month n...  Anxiety

Columns: ['Unnamed: 0', 'statement', 'status']

Label distribution:
status
Normal                  16351
Depression              15404
Suicidal                10652
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64


In [ ]:
# MentalChat16K
from datasets import load_dataset
mc16k = load_dataset("ShenLab/MentalChat16K")
df_mental = mc16k['train'].to_pandas()
print("MentalChat16K columns:", df_mental.columns.tolist())

README.md:   0%|          | 0.00/3.95k [00:00<?, ?B/s]

Interview_Data_6K.csv: reconstructing file:   0%|          |  0.00B / 13.6MB            

Interview_Data_6K.csv: downloading bytes:           |  0.00B            

Synthetic_Data_10K.csv: reconstructing file:   0%|          |  0.00B / 32.8MB            

Synthetic_Data_10K.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16084 [00:00<?, ? examples/s]

MentalChat16K columns: ['instruction', 'input', 'output']


In [ ]:
# ============================================================
# CELL 3 — Check actual column names first
# ============================================================
print("Sentiment columns:", df_sentiment.columns.tolist())

Sentiment columns: ['Unnamed: 0', 'statement', 'status']


In [ ]:
# ============================================================
# CELL 4 — FIXED standardize datasets
# ============================================================

# --- Sentiment dataset (columns: statement, status) ---
df_sentiment_std = pd.DataFrame({
    'text'  : df_sentiment['statement'],
    'label' : df_sentiment['status']
}).dropna()

print("Sentiment standardized:", df_sentiment_std.shape)
print("Labels:\n", df_sentiment_std['label'].value_counts())

Sentiment standardized: (52680, 2)
Labels:
 label
Normal                  16343
Depression              15404
Suicidal                10651
Anxiety                  3841
Bipolar                  2777
Stress                   2587
Personality disorder     1077
Name: count, dtype: int64


In [ ]:
# ============================================================
# CELL 6 — Proceed with Sentiment dataset ONLY (has labels)
# ============================================================
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df_train_text = df_sentiment_std.copy()

print("Training dataset:", df_train_text.shape)
print(df_train_text['label'].value_counts())

le_text = LabelEncoder()
df_train_text['label_encoded'] = le_text.fit_transform(df_train_text['label'])

print("\nEncoding map:")
for i, c in enumerate(le_text.classes_):
    print(f"  {i} → {c}")

X = df_train_text['text'].tolist()
y = df_train_text['label_encoded'].tolist()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Training dataset: (52680, 2)
label
Normal                  16343
Depression              15404
Suicidal                10651
Anxiety                  3841
Bipolar                  2777
Stress                   2587
Personality disorder     1077
Name: count, dtype: int64

Encoding map:
  0 → Anxiety
  1 → Bipolar
  2 → Depression
  3 → Normal
  4 → Personality disorder
  5 → Stress
  6 → Suicidal

Train: 42144 | Val: 5268 | Test: 5268


In [ ]:
# ============================================================
# CELL 6 FIXED — Ensure all text values are clean strings
# ============================================================

# Check for problematic values first
print("Checking data types in X_train...")
non_string_count = sum(1 for x in X_train if not isinstance(x, str))
print(f"Non-string values found: {non_string_count}")

# Check for None/NaN
none_count = sum(1 for x in X_train if x is None)
print(f"None values: {none_count}")

# Show a few examples of bad entries
bad_examples = [x for x in X_train if not isinstance(x, str)][:5]
print("Bad examples:", bad_examples)

Checking data types in X_train...
Non-string values found: 0
None values: 0
Bad examples: []


In [ ]:
# ============================================================
# CELL 6B — Force clean all text data
# ============================================================

# Clean train
X_train = [str(x) if x is not None else "" for x in X_train]
X_train = [x.strip() for x in X_train]
X_train = [x if x else "empty" for x in X_train]  # replace empty with placeholder

# Clean val
X_val = [str(x) if x is not None else "" for x in X_val]
X_val = [x.strip() for x in X_val]
X_val = [x if x else "empty" for x in X_val]

# Clean test
X_test = [str(x) if x is not None else "" for x in X_test]
X_test = [x.strip() for x in X_test]
X_test = [x if x else "empty" for x in X_test]

print("Cleaned X_train sample:", X_train[:2])
print("Any remaining non-strings:", sum(1 for x in X_train if not isinstance(x, str)))

Cleaned X_train sample: ['The only thing i keep myself alive for is self pityful hope in therapy. I am extremely irresponsible, often skip meds and do not bring test results on time. Obviously, they are not working. My therapist and I do not have chemistry and I ever only had a slither of hope when she was confident in my first diagnosis during my first session. I have seen her numerous times and tried various medication but nothing seems to work, even when I take it responsibly but I guess i do not endure enough long. I am sabotaging my own promise. Most recently I was on olanzapine, aripiprazole and eftil. She proposed to try lithium salts but, obviously, i skipped the necessary tests because of how much of a failure I am.I have nothing else to try for. I will talk to her one last time and if nothing promising happens, that is it. I promised before ending it all to give one last pityful attempt to therapy, and its not working.', 'malaysia is getting more tenacious guys........']
Any 

In [ ]:
from huggingface_hub import login
login(token="REMOVED_HF_TOKEN_SEE_ENV")
print("Logged in successfully")

Logged in successfully


In [ ]:
# ============================================================
# CELL 7 — Tokenize with MentalBERT (HF login already done)
# ============================================================
from transformers import AutoTokenizer

MODEL_NAME = "mental/mental-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token = True
)
print("Tokenizer loaded:", MODEL_NAME)

def tokenize_batch(texts, max_len=128):
    return tokenizer(
        texts,
        padding        = True,
        truncation     = True,
        max_length     = max_len,
        return_tensors = 'pt'
    )

print("Tokenizing train set...")
train_encodings = tokenize_batch(X_train)

print("Tokenizing val set...")
val_encodings   = tokenize_batch(X_val)

print("Tokenizing test set...")
test_encodings  = tokenize_batch(X_test)

print("\nTokenization done.")
print("Train shape:", train_encodings['input_ids'].shape)

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded: mental/mental-bert-base-uncased
Tokenizing train set...
Tokenizing val set...
Tokenizing test set...

Tokenization done.
Train shape: torch.Size([42144, 128])


In [ ]:
# ============================================================
# CELL 8 — Create PyTorch Dataset + DataLoaders
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader

class MentalHealthDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: val[idx]
            for key, val in self.encodings.items()
        }
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = MentalHealthDataset(train_encodings, y_train)
val_dataset   = MentalHealthDataset(val_encodings,   y_val)
test_dataset  = MentalHealthDataset(test_encodings,  y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16)
test_loader  = DataLoader(test_dataset,  batch_size=16)

print("DataLoaders ready.")
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

DataLoaders ready.
Train batches : 2634
Val batches   : 330
Test batches  : 330


In [ ]:
# ============================================================
# CELL 9 — Check GPU in Colab
# ============================================================
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memory:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")
else:
    print("No GPU — go to Runtime > Change Runtime Type > T4 GPU")

Device: cuda
GPU: Tesla T4
Memory: 15.6 GB


In [ ]:
# ============================================================
# CELL 10 — Load MentalBERT and Train
# ============================================================
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
import time

num_classes = len(le_text.classes_)
print("Number of classes:", num_classes)
print("Classes:", list(le_text.classes_))

model_text = AutoModelForSequenceClassification.from_pretrained(
    "mental/mental-bert-base-uncased",
    num_labels = num_classes,
    token      = True
)
model_text.to(device)

optimizer = AdamW(model_text.parameters(), lr=2e-5)
EPOCHS    = 3

print(f"\nStarting training on {device}...")
print(f"Total batches per epoch: {len(train_loader)}")
print("="*55)

for epoch in range(EPOCHS):
    model_text.train()
    total_loss = 0
    correct    = 0
    total      = 0
    start_time = time.time()

    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model_text(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            labels         = labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds       = outputs.logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        if (batch_idx + 1) % 200 == 0:
            elapsed   = time.time() - start_time
            remaining = (elapsed/(batch_idx+1)) * (len(train_loader)-batch_idx-1)
            print(f"  Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f} | "
                  f"ETA: {remaining/60:.1f} min")

    # Validation
    model_text.eval()
    val_correct = 0
    val_total   = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)
            outputs        = model_text(
                input_ids      = input_ids,
                attention_mask = attention_mask
            )
            preds        = outputs.logits.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)

    train_acc  = correct / total
    val_acc    = val_correct / val_total
    epoch_time = (time.time() - start_time) / 60

    print(f"\n{'='*55}")
    print(f"Epoch {epoch+1}/{EPOCHS} Complete")
    print(f"  Train Accuracy : {train_acc:.4f}")
    print(f"  Val Accuracy   : {val_acc:.4f}")
    print(f"  Avg Loss       : {total_loss/len(train_loader):.4f}")
    print(f"  Time           : {epoch_time:.1f} min")
    print(f"{'='*55}\n")

    torch.save(model_text.state_dict(),
               f"text_model_epoch{epoch+1}.pt")
    print(f"Saved: text_model_epoch{epoch+1}.pt")

print("\nTraining complete!")
torch.save(model_text.state_dict(), "text_model_final.pt")
print("Saved: text_model_final.pt")

Number of classes: 7
Classes: ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder', 'Stress', 'Suicidal']


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 


Starting training on cuda...
Total batches per epoch: 2634


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

  Batch 200/2634 | Loss: 0.8500 | ETA: 13.5 min
  Batch 400/2634 | Loss: 0.3040 | ETA: 12.8 min
  Batch 600/2634 | Loss: 0.7524 | ETA: 11.9 min
  Batch 800/2634 | Loss: 0.5824 | ETA: 10.9 min
  Batch 1000/2634 | Loss: 0.4008 | ETA: 9.7 min
  Batch 1200/2634 | Loss: 0.3279 | ETA: 8.6 min
  Batch 1400/2634 | Loss: 0.3952 | ETA: 7.4 min
  Batch 1600/2634 | Loss: 0.3214 | ETA: 6.2 min
  Batch 1800/2634 | Loss: 0.2250 | ETA: 5.0 min
  Batch 2000/2634 | Loss: 0.9361 | ETA: 3.8 min
  Batch 2200/2634 | Loss: 0.5557 | ETA: 2.6 min
  Batch 2400/2634 | Loss: 0.4339 | ETA: 1.4 min
  Batch 2600/2634 | Loss: 0.1952 | ETA: 0.2 min

Epoch 1/3 Complete
  Train Accuracy : 0.7892
  Val Accuracy   : 0.8242
  Avg Loss       : 0.5324
  Time           : 16.6 min

Saved: text_model_epoch1.pt
  Batch 200/2634 | Loss: 0.3594 | ETA: 15.0 min
  Batch 400/2634 | Loss: 0.2492 | ETA: 13.7 min
  Batch 600/2634 | Loss: 0.0924 | ETA: 12.4 min
  Batch 800/2634 | Loss: 0.3223 | ETA: 11.2 min
  Batch 1000/2634 | Loss: 0.6

In [ ]:
# ============================================================
# CELL 11 — Evaluate on test set
# ============================================================
from sklearn.metrics import classification_report, accuracy_score, f1_score

model_text.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels']

        outputs = model_text(
            input_ids      = input_ids,
            attention_mask = attention_mask
        )
        preds = outputs.logits.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

print("="*55)
print("TEXT MODEL — FINAL TEST RESULTS")
print("="*55)
print(f"Accuracy : {accuracy_score(all_labels, all_preds):.4f}")
print(f"F1 Macro : {f1_score(all_labels, all_preds, average='macro'):.4f}")
print(f"F1 Weight: {f1_score(all_labels, all_preds, average='weighted'):.4f}")
print()
print(classification_report(
    all_labels, all_preds,
    target_names = le_text.classes_
))

TEXT MODEL — FINAL TEST RESULTS
Accuracy : 0.8426
F1 Macro : 0.8290
F1 Weight: 0.8421

                      precision    recall  f1-score   support

             Anxiety       0.90      0.86      0.88       384
             Bipolar       0.88      0.85      0.87       278
          Depression       0.78      0.81      0.79      1540
              Normal       0.95      0.97      0.96      1635
Personality disorder       0.85      0.72      0.78       107
              Stress       0.79      0.80      0.79       259
            Suicidal       0.75      0.71      0.73      1065

            accuracy                           0.84      5268
           macro avg       0.84      0.82      0.83      5268
        weighted avg       0.84      0.84      0.84      5268

